In [3]:
import pandas as pd

In [4]:
caltech = pd.read_json(
    "acndata_download/acndata_caltech_sessions.jsonl",
    lines=True
)

jpl = pd.read_json(
    "acndata_download/acndata_jpl_sessions.jsonl",
    lines=True
)

caltech["site"] = "caltech"
jpl["site"] = "jpl"

df = pd.concat([caltech, jpl], ignore_index=True)

df.head()

,_id,userInputs,userID,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,site
0,61550519f9af8b76960e169e,"[{'userID': 19055, 'milesRequested': 100, 'WhP...",19055.0,2_39_81_4550_2021-09-14 01:52:31.129163,2-39-81-4550,11900388,2,39,"Tue, 14 Sep 2021 01:52:37 GMT","Tue, 14 Sep 2021 03:05:10 GMT",45.064,NaN,America/Los_Angeles,caltech
1,61550519f9af8b76960e169d,"[{'userID': 1082, 'milesRequested': 20, 'WhPer...",1082.0,2_39_91_437_2021-09-13 23:10:59.528292,2-39-91-437,CA-317,2,39,"Mon, 13 Sep 2021 23:11:12 GMT","Tue, 14 Sep 2021 01:43:11 GMT",2.018,"Tue, 14 Sep 2021 00:13:35 GMT",America/Los_Angeles,caltech
2,61550519f9af8b76960e169c,"[{'userID': 3905, 'milesRequested': 100, 'WhPe...",3905.0,2_39_81_4550_2021-09-13 22:33:04.543952,2-39-81-4550,11900388,2,39,"Mon, 13 Sep 2021 22:33:07 GMT","Mon, 13 Sep 2021 23:06:55 GMT",17.720,NaN,America/Los_Angeles,caltech
3,61550519f9af8b76960e169b,"[{'userID': 6481, 'milesRequested': 25, 'WhPer...",6481.0,2_39_123_23_2021-09-13 21:16:44.026068,2-39-123-23,CA-313,2,39,"Mon, 13 Sep 2021 21:17:04 GMT","Tue, 14 Sep 2021 01:01:49 GMT",6.715,"Mon, 13 Sep 2021 23:18:07 GMT",America/Los_Angeles,caltech
4,61550519f9af8b76960e169a,"[{'userID': 431, 'milesRequested': 100, 'WhPer...",431.0,2_39_89_25_2021-09-13 21:12:53.318460,2-39-89-25,CA-315,2,39,"Mon, 13 Sep 2021 21:12:53 GMT","Tue, 14 Sep 2021 00:25:36 GMT",2.285,"Mon, 13 Sep 2021 21:41:31 GMT",America/Los_Angeles,caltech


In [ ]:
**Check Missing and Duplicat

In [5]:
df.info()

display(df.isnull().sum().sort_values(ascending=False))

df.duplicated(subset=["site", "sessionID"]).sum()

<class 'pandas.DataFrame'>
RangeIndex: 65062 entries, 0 to 65061
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   _id               65062 non-null  str    
 1   userInputs        47847 non-null  object 
 2   userID            47847 non-null  float64
 3   sessionID         65062 non-null  str    
 4   stationID         65062 non-null  str    
 5   spaceID           65062 non-null  str    
 6   siteID            65062 non-null  int64  
 7   clusterID         65062 non-null  int64  
 8   connectionTime    65062 non-null  str    
 9   disconnectTime    65062 non-null  str    
 10  kWhDelivered      65062 non-null  float64
 11  doneChargingTime  60970 non-null  str    
 12  timezone          65062 non-null  str    
 13  site              65062 non-null  str    
dtypes: float64(2), int64(2), object(1), str(9)
memory usage: 6.9+ MB


userInputs          17215
userID              17215
doneChargingTime     4092
_id                     0
stationID               0
sessionID               0
spaceID                 0
siteID                  0
connectionTime          0
clusterID               0
disconnectTime          0
kWhDelivered            0
timezone                0
site                    0
dtype: int64

np.int64(0)

**Data is being cleaned, to make sure all data are true values**

In [8]:
# List the columns that contain date and time values.
date_columns = [
    "connectionTime",
    "disconnectTime",
    "doneChargingTime"
]

# Convert each date column into a proper datetime format.
for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce", 
        utc=True          
    )

# Convert the delivered energy column into numeric values.
df["kWhDelivered"] = pd.to_numeric(
    df["kWhDelivered"],
    errors="coerce"  
)

# Remove sessions missing the essential values needed for the study.
df = df.dropna(
    subset=[
        "sessionID",
        "connectionTime",
        "kWhDelivered"
    ]
)

# Remove duplicate sessions within each site.
df = df.drop_duplicates(
    subset=["site", "sessionID"],
    keep="first"  # Keep the first copy of each duplicated session.
)

# Remove impossible negative energy values.
df = df[df["kWhDelivered"] >= 0]

# Keep sessions with no disconnection time or with a valid time order.
df = df[
    df["disconnectTime"].isna()
    | (df["disconnectTime"] >= df["connectionTime"])
]

# Convert connection time from UTC to the local California timezone.
df["connectionTimeLocal"] = (
    df["connectionTime"]
    .dt.tz_convert("America/Los_Angeles")
)

# Extract only the local connection date for daily aggregation later.
df["date"] = (
    df["connectionTimeLocal"]
    .dt.date
)

# Reset the row numbers after removing invalid and duplicate records.
df = df.reset_index(drop=True)

In [9]:
cleaning_summary = pd.Series({
    "rows_after_cleaning": len(df),
    "missing_required_values": df[
        ["sessionID", "connectionTime", "kWhDelivered"]
    ].isnull().sum().sum(),
    "duplicate_sessions": df.duplicated(
        subset=["site", "sessionID"]
    ).sum(),
    "negative_energy_values": (
        df["kWhDelivered"] < 0
    ).sum()
})

display(cleaning_summary)

df.head()

rows_after_cleaning        65062
missing_required_values        0
duplicate_sessions             0
negative_energy_values         0
dtype: int64

,_id,userInputs,userID,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,site,connectionTimeLocal,date
0,61550519f9af8b76960e169e,"[{'userID': 19055, 'milesRequested': 100, 'WhP...",19055.0,2_39_81_4550_2021-09-14 01:52:31.129163,2-39-81-4550,11900388,2,39,2021-09-14 01:52:37+00:00,2021-09-14 03:05:10+00:00,45.064,NaT,America/Los_Angeles,caltech,2021-09-13 18:52:37-07:00,2021-09-13
1,61550519f9af8b76960e169d,"[{'userID': 1082, 'milesRequested': 20, 'WhPer...",1082.0,2_39_91_437_2021-09-13 23:10:59.528292,2-39-91-437,CA-317,2,39,2021-09-13 23:11:12+00:00,2021-09-14 01:43:11+00:00,2.018,2021-09-14 00:13:35+00:00,America/Los_Angeles,caltech,2021-09-13 16:11:12-07:00,2021-09-13
2,61550519f9af8b76960e169c,"[{'userID': 3905, 'milesRequested': 100, 'WhPe...",3905.0,2_39_81_4550_2021-09-13 22:33:04.543952,2-39-81-4550,11900388,2,39,2021-09-13 22:33:07+00:00,2021-09-13 23:06:55+00:00,17.720,NaT,America/Los_Angeles,caltech,2021-09-13 15:33:07-07:00,2021-09-13
3,61550519f9af8b76960e169b,"[{'userID': 6481, 'milesRequested': 25, 'WhPer...",6481.0,2_39_123_23_2021-09-13 21:16:44.026068,2-39-123-23,CA-313,2,39,2021-09-13 21:17:04+00:00,2021-09-14 01:01:49+00:00,6.715,2021-09-13 23:18:07+00:00,America/Los_Angeles,caltech,2021-09-13 14:17:04-07:00,2021-09-13
4,61550519f9af8b76960e169a,"[{'userID': 431, 'milesRequested': 100, 'WhPer...",431.0,2_39_89_25_2021-09-13 21:12:53.318460,2-39-89-25,CA-315,2,39,2021-09-13 21:12:53+00:00,2021-09-14 00:25:36+00:00,2.285,2021-09-13 21:41:31+00:00,America/Los_Angeles,caltech,2021-09-13 14:12:53-07:00,2021-09-13


**Charging sessions are combined by site and date to calculate daily energy demand, then arranged chronologically for forecasting.**

In [10]:
# Combine all charging sessions that occurred on the same site and date.
daily_df = (
    df.groupby(["site", "date"], as_index=False)
    .agg(
        dailyEnergy=("kWhDelivered", "sum"),
        sessionCount=("sessionID", "count"),
        activeStations=("stationID", "nunique"),
        averageEnergyPerSession=("kWhDelivered", "mean")
    )
)

# Convert the date column into datetime format.
daily_df["date"] = pd.to_datetime(daily_df["date"])

# Arrange the data chronologically for each site.
daily_df = daily_df.sort_values(
    ["site", "date"]
).reset_index(drop=True)

daily_df.head()

,site,date,dailyEnergy,sessionCount,activeStations,averageEnergyPerSession
0,caltech,2018-04-25,447.938,60,36,7.465633
1,caltech,2018-04-26,338.813,47,37,7.208787
2,caltech,2018-04-27,572.427,51,34,11.224059
3,caltech,2018-04-28,341.136,30,16,11.371200
4,caltech,2018-04-29,266.255,29,17,9.181207


In [11]:
# Check the structure and missing values of the daily dataset.
daily_df.info()

display(daily_df.isnull().sum())

daily_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2037 entries, 0 to 2036
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype        
---  ------                   --------------  -----        
 0   site                     2037 non-null   str          
 1   date                     2037 non-null   datetime64[s]
 2   dailyEnergy              2037 non-null   float64      
 3   sessionCount             2037 non-null   int64        
 4   activeStations           2037 non-null   int64        
 5   averageEnergyPerSession  2037 non-null   float64      
dtypes: datetime64[s](1), float64(2), int64(2), str(1)
memory usage: 95.6 KB


site                       0
date                       0
dailyEnergy                0
sessionCount               0
activeStations             0
averageEnergyPerSession    0
dtype: int64

,site,date,dailyEnergy,sessionCount,activeStations,averageEnergyPerSession
0,caltech,2018-04-25,447.938,60,36,7.465633
1,caltech,2018-04-26,338.813,47,37,7.208787
2,caltech,2018-04-27,572.427,51,34,11.224059
3,caltech,2018-04-28,341.136,30,16,11.371200
4,caltech,2018-04-29,266.255,29,17,9.181207


In [12]:
# Create a list that will store the complete calendar for each site.
complete_data = []

# Process Caltech and JPL separately.
for site in daily_df["site"].unique():

    # Select the daily records belonging to the current site.
    site_data = daily_df[
        daily_df["site"] == site
    ].copy()

    # Create every date between the site's first and last date.
    all_dates = pd.DataFrame({
        "date": pd.date_range(
            start=site_data["date"].min(),
            end=site_data["date"].max(),
            freq="D"
        )
    })

    # Add the site name to the calendar.
    all_dates["site"] = site

    # Combine the complete calendar with the available daily records.
    site_data = all_dates.merge(
        site_data,
        on=["site", "date"],
        how="left"
    )

    # Mark dates that do not have charging records.
    site_data["missingDay"] = (
        site_data["dailyEnergy"].isna()
    )

    # Add the site's calendar to the list.
    complete_data.append(site_data)

# Combine the completed calendars of Caltech and JPL.
daily_complete_df = pd.concat(
    complete_data,
    ignore_index=True
)

In [13]:
# Count the number of missing calendar dates for each site.
display(
    daily_complete_df
    .groupby("site")["missingDay"]
    .sum()
)

# Display some of the missing dates.
daily_complete_df[
    daily_complete_df["missingDay"]
].head(20)

site
caltech    154
jpl        152
Name: missingDay, dtype: int64

,date,site,dailyEnergy,sessionCount,activeStations,averageEnergyPerSession,missingDay
609,2019-12-25,caltech,NaN,NaN,NaN,NaN,True
695,2020-03-20,caltech,NaN,NaN,NaN,NaN,True
696,2020-03-21,caltech,NaN,NaN,NaN,NaN,True
701,2020-03-26,caltech,NaN,NaN,NaN,NaN,True
705,2020-03-30,caltech,NaN,NaN,NaN,NaN,True
707,2020-04-01,caltech,NaN,NaN,NaN,NaN,True
712,2020-04-06,caltech,NaN,NaN,NaN,NaN,True
714,2020-04-08,caltech,NaN,NaN,NaN,NaN,True
715,2020-04-09,caltech,NaN,NaN,NaN,NaN,True
721,2020-04-15,caltech,NaN,NaN,NaN,NaN,True
